# Stage 1 -- Patching (Chipping Rasters to Disk)

**Purpose:** Chip the full-resolution merged GeoTIFFs into 256x256 and 512x512 patches, saving each chip as an individual file on disk.

**Study Area:** Kalimantan, Indonesia  

---

## Design Principles

1. **Bitemporal alignment first.** ChangeFormer requires paired images at T1 and
   T2, plus a change mask. We chip all three from the **same spatial window** so
   that every chip triplet (image_T1, image_T2, mask) is pixel-aligned by
   construction.

2. **Raw bands only at this stage.** We chip the 4-band raster (B2, B3, B4, B8)
   or the per-channel rasters (NGB, RGB) as-is. Band compositions (NDVI, EVI)
   are derived in Stage 2, guaranteeing they share identical spatial footprints.

3. **Minimal filtering here.** Only reject chips with excessive nodata. The
   change-class filtering (single-class, <10% threshold) is applied in Stage 2
   because filtering requirements may change during experimentation.

4. **Disk-based output.** Each chip is saved as a compressed `.tif` file.
   This avoids holding the entire dataset in memory.

In [14]:
import os
import json
import shutil
import logging
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional

import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.merge import merge
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger(__name__)

## Configuration

In [17]:
@dataclass
class PatchingConfig:
    """Configuration for the patching stage."""

    # --- Input directory (raw GEE exports) ---
    raw_dir: Path = Path("Dataset/GEE_Kalimantan_v2")

    # --- Intermediate directory (merged tiles) ---
    merged_dir: Path = Path("Dataset/GEE_Kalimantan_Merged")

    # --- Output directory (chips on disk) ---
    chips_dir: Path = Path("Dataset/GEE_Kalimantan_Chips")

    # --- Temporal ---
    years: List[int] = field(default_factory=lambda: [2021, 2022, 2023, 2024])

    # Bitemporal pairs: (T1, T2)
    year_pairs: List[Tuple[int, int]] = field(
        default_factory=lambda: [(2021, 2022), (2022, 2023), (2023, 2024)]
    )

    # --- Chip sizes (paper Section II-D) ---
    chip_sizes: List[int] = field(default_factory=lambda: [256, 512])

    # --- Overlap ---
    overlap: float = 0.25  # 0.0 = no overlap, 0.25 = 25% overlap

    # --- Nodata filtering ---
    # Reject chips where more than this fraction of pixels are nodata/zero
    min_valid_ratio: float = 0.80

    def __post_init__(self):
        self.merged_dir.mkdir(parents=True, exist_ok=True)
        self.chips_dir.mkdir(parents=True, exist_ok=True)


config = PatchingConfig()

log.info("Patching Configuration")
log.info(f"  Raw data  : {config.raw_dir}")
log.info(f"  Merged  : {config.merged_dir}")
log.info(f"  Chips out : {config.chips_dir}")
log.info(f"  Year pairs: {config.year_pairs}")
log.info(f"  Chip sizes: {config.chip_sizes}")
log.info(f"  Min valid : {config.min_valid_ratio:.0%}")

21:04:44 | INFO | Patching Configuration
21:04:44 | INFO |   Raw data  : Dataset\GEE_Kalimantan_v2
21:04:44 | INFO |   Merged  : Dataset\GEE_Kalimantan_Merged
21:04:44 | INFO |   Chips out : Dataset\GEE_Kalimantan_Chips
21:04:44 | INFO |   Year pairs: [(2021, 2022), (2022, 2023), (2023, 2024)]
21:04:44 | INFO |   Chip sizes: [256, 512]
21:04:44 | INFO |   Min valid : 80%


## Step 1 -- Tile Discovery and Merging

GEE splits large exports into multiple tiles. We merge them into a single
seamless raster per year so that chip grid positions are consistent across
all bands and years.

In [18]:
def discover_tiles(raw_dir: Path, prefix: str) -> List[Path]:
    """Find all GeoTIFF tiles matching a naming prefix."""
    tiles = sorted(raw_dir.glob(f"{prefix}*.tif"))
    return tiles


def merge_tiles(tile_paths: List[Path], output_path: Path) -> Path:
    """
    Merge multiple GEE export tiles into a single GeoTIFF.
    Skips if the output already exists.
    """
    if output_path.exists():
        log.info(f"  [skip] Already exists: {output_path.name}")
        return output_path

    if len(tile_paths) == 0:
        log.warning(f"  No tiles found for {output_path.name}")
        return None

    if len(tile_paths) == 1:
        shutil.copy2(tile_paths[0], output_path)
        size_mb = output_path.stat().st_size / (1024 ** 2)
        log.info(f"  Single tile -> {output_path.name} ({size_mb:.1f} MB)")
        return output_path

    src_files = [rasterio.open(p) for p in tile_paths]
    try:
        mosaic, out_transform = merge(src_files)
    finally:
        for s in src_files:
            s.close()

    profile = rasterio.open(tile_paths[0]).profile.copy()
    profile.update(
        height=mosaic.shape[1],
        width=mosaic.shape[2],
        transform=out_transform,
        compress='lzw',
    )

    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(mosaic)

    size_mb = output_path.stat().st_size / (1024 ** 2)
    log.info(
        f"  Merged {len(tile_paths)} tiles -> {output_path.name} "
        f"({mosaic.shape[2]}x{mosaic.shape[1]}, {size_mb:.1f} MB)"
    )
    return output_path

In [19]:
def detect_layout(raw_dir: Path, years: List[int]) -> str:
    """
    Detect data layout.
      Layout A: single multi-band export (Kalimantan_S2_Bands_YYYY)
      Layout B: separate channel exports (Kalimantan_S2_NGB_YYYY, etc.)
    """
    if list(raw_dir.glob(f"Kalimantan_S2_Bands_{years[0]}*.tif")):
        return 'A'
    elif list(raw_dir.glob(f"Kalimantan_S2_NGB_{years[0]}*.tif")):
        return 'B'
    else:
        raise FileNotFoundError(
            f"No Sentinel-2 exports found in {raw_dir}. "
            "Check filenames match expected patterns."
        )


def run_merge(config: PatchingConfig) -> Dict:
    """
    Merge all tiles and return a registry of merged file paths.
    """
    layout = detect_layout(config.raw_dir, config.years)
    log.info(f"Data layout: {'A (multi-band)' if layout == 'A' else 'B (per-channel)'}")

    reg = {'layout': layout, 'imagery': {}, 'masks': {}, 'forest2000': None}

    # --- Sentinel-2 imagery ---
    log.info("\nMerging Sentinel-2 imagery...")
    for year in config.years:
        if layout == 'A':
            tiles = discover_tiles(config.raw_dir, f"Kalimantan_S2_Bands_{year}")
            out = config.merged_dir / f"S2_Bands_{year}.tif"
            reg['imagery'][year] = merge_tiles(tiles, out)
        else:
            reg['imagery'][year] = {}
            for channel in ['NGB', 'RGB', 'NDVI', 'EVI']:
                tiles = discover_tiles(config.raw_dir, f"Kalimantan_S2_{channel}_{year}")
                if tiles:
                    out = config.merged_dir / f"S2_{channel}_{year}.tif"
                    reg['imagery'][year][channel] = merge_tiles(tiles, out)

    # --- Ground truth masks ---
    log.info("\nMerging ground truth masks...")

    # Bitemporal change masks (from corrected fetching)
    for i in range(len(config.years) - 1):
        t1, t2 = config.years[i], config.years[i + 1]
        key = f"{t1}_{t2}"
        tiles = discover_tiles(config.raw_dir, f"Kalimantan_Hansen_Change_{key}")
        if tiles:
            out = config.merged_dir / f"Hansen_Change_{key}.tif"
            reg['masks'][key] = merge_tiles(tiles, out)

    # Annual loss masks (from original fetching -- fallback)
    for year in config.years:
        tiles = discover_tiles(config.raw_dir, f"Kalimantan_Hansen_Loss_{year}")
        if tiles:
            out = config.merged_dir / f"Hansen_Loss_{year}.tif"
            reg['masks'][f"annual_{year}"] = merge_tiles(tiles, out)

    # Forest 2000
    tiles = discover_tiles(config.raw_dir, "Kalimantan_Hansen_Forest2000")
    if tiles:
        out = config.merged_dir / "Hansen_Forest2000.tif"
        reg['forest2000'] = merge_tiles(tiles, out)

    return reg


log.info("=" * 70)
log.info("STEP 1: Tile discovery and merging")
log.info("=" * 70)
registry = run_merge(config)

21:04:48 | INFO | ======================================================================
21:04:48 | INFO | STEP 1: Tile discovery and merging
21:04:48 | INFO | ======================================================================


21:04:48 | INFO | Data layout: A (multi-band)
21:04:48 | INFO | 
Merging Sentinel-2 imagery...
21:07:42 | INFO |   Merged 2 tiles -> S2_Bands_2021.tif (16699x14474, 2105.2 MB)
21:09:33 | INFO |   Merged 2 tiles -> S2_Bands_2022.tif (16699x14474, 1948.8 MB)
21:11:28 | INFO |   Merged 2 tiles -> S2_Bands_2023.tif (16699x14474, 1780.7 MB)
21:13:19 | INFO |   Merged 2 tiles -> S2_Bands_2024.tif (16699x14474, 2044.4 MB)
21:13:20 | INFO | 
Merging ground truth masks...
21:13:21 | INFO |   Merged 2 tiles -> Hansen_Change_2021_2022.tif (16699x14474, 2.0 MB)
21:13:22 | INFO |   Merged 2 tiles -> Hansen_Change_2022_2023.tif (16699x14474, 2.2 MB)
21:13:24 | INFO |   Merged 2 tiles -> Hansen_Change_2023_2024.tif (16699x14474, 2.1 MB)
21:13:25 | INFO |   Merged 2 tiles -> Hansen_Loss_2021.tif (16699x14474, 1.9 MB)
21:13:26 | INFO |   Merged 2 tiles -> Hansen_Loss_2022.tif (16699x14474, 2.0 MB)
21:13:27 | INFO |   Merged 2 tiles -> Hansen_Loss_2023.tif (16699x14474, 2.2 MB)
21:13:28 | INFO |   Merge

## Step 2 -- Resolve Mask Paths for Each Year Pair

For each bitemporal pair (T1, T2), we need a ground truth change mask.
Priority:
1. Bitemporal change mask (`Hansen_Change_T1_T2`) -- from corrected export
2. Annual loss mask for T2 (`Hansen_Loss_T2`) -- from original export

The annual loss mask for year T2 captures deforestation that occurred in T2,
which is a reasonable proxy for the bitemporal change between T1 and T2.

In [20]:
def resolve_mask_path(registry: Dict, t1: int, t2: int) -> Optional[Path]:
    """Resolve the ground truth mask path for a year pair."""
    masks = registry['masks']

    # Priority 1: bitemporal change mask
    key_bt = f"{t1}_{t2}"
    if key_bt in masks and masks[key_bt] is not None:
        return masks[key_bt]

    # Priority 2: annual loss for T2
    key_ann = f"annual_{t2}"
    if key_ann in masks and masks[key_ann] is not None:
        log.info(f"  Using annual loss mask for {t2} as proxy for {t1}->{t2}")
        return masks[key_ann]

    log.warning(f"  No mask found for pair {t1}->{t2}")
    return None


# Verify all pairs have masks
log.info("\nMask resolution:")
for t1, t2 in config.year_pairs:
    mp = resolve_mask_path(registry, t1, t2)
    if mp:
        log.info(f"  {t1} -> {t2}: {mp.name}")
    else:
        log.warning(f"  {t1} -> {t2}: MISSING")

21:16:43 | INFO | 
Mask resolution:
21:16:43 | INFO |   2021 -> 2022: Hansen_Change_2021_2022.tif
21:16:43 | INFO |   2022 -> 2023: Hansen_Change_2022_2023.tif
21:16:43 | INFO |   2023 -> 2024: Hansen_Change_2023_2024.tif


## Step 3 -- Patching (Core Chipping Loop)

For each year pair and chip size:

1. Open the T1 imagery, T2 imagery, and change mask rasters.
2. Iterate over a grid of non-overlapping (or overlapping) windows.
3. Read the chip from all three rasters at the same window position.
4. Apply nodata filtering.
5. Save valid chips to disk.

### Output structure

```
chips_dir/
    256/
        2021_2022/
            images_t1/
                chip_r0000_c0000.tif   (bands depend on layout)
                chip_r0000_c0001.tif
                ...
            images_t2/
                chip_r0000_c0000.tif
                ...
            masks/
                chip_r0000_c0000.tif
                ...
            chip_index.json
    512/
        ...
```

This structure keeps T1, T2, and mask chips perfectly aligned by filename.

In [21]:
def get_imagery_path(registry: Dict, year: int) -> Path:
    """
    Get the primary imagery raster path for a given year.
    Layout A: single multi-band raster.
    Layout B: NGB raster (contains B8, B3, B2 -- the primary composition).
    """
    if registry['layout'] == 'A':
        return registry['imagery'][year]
    else:
        # For layout B, use NGB as the primary raster for chipping
        return registry['imagery'][year]['NGB']


def get_raster_dimensions(path: Path) -> Tuple[int, int, int]:
    """Return (height, width, n_bands) of a raster."""
    with rasterio.open(path) as src:
        return src.height, src.width, src.count


def save_chip(
    data: np.ndarray,
    output_path: Path,
    profile: dict,
    window: Window,
    src_transform,
):
    """
    Save a chip to disk as a GeoTIFF, preserving georeference.

    Parameters
    ----------
    data : np.ndarray
        Shape (bands, height, width) for imagery, or (height, width) for masks.
    output_path : Path
    profile : dict
        Base rasterio profile (crs, dtype, etc.).
    window : Window
        The rasterio window used to extract this chip.
    src_transform
        The transform of the source raster.
    """
    if data.ndim == 2:
        data = data[np.newaxis, :, :]

    chip_transform = rasterio.windows.transform(window, src_transform)

    chip_profile = profile.copy()
    chip_profile.update(
        height=data.shape[1],
        width=data.shape[2],
        count=data.shape[0],
        transform=chip_transform,
        compress='lzw',
        dtype=data.dtype,
    )

    with rasterio.open(output_path, 'w', **chip_profile) as dst:
        dst.write(data)

In [22]:
def patch_year_pair(
    t1: int,
    t2: int,
    chip_size: int,
    registry: Dict,
    config: PatchingConfig,
) -> Dict:
    """
    Chip a single year pair at a given chip size.

    Reads T1 imagery, T2 imagery, and the change mask from the same
    spatial window, filters for nodata, and saves valid chips to disk.

    Returns
    -------
    dict : Chipping statistics and chip index.
    """
    pair_key = f"{t1}_{t2}"

    # Resolve paths
    t1_path = get_imagery_path(registry, t1)
    t2_path = get_imagery_path(registry, t2)
    mask_path = resolve_mask_path(registry, t1, t2)

    if t1_path is None or t2_path is None:
        log.warning(f"  Missing imagery for {pair_key}, skipping.")
        return {'created': 0}
    if mask_path is None:
        log.warning(f"  Missing mask for {pair_key}, skipping.")
        return {'created': 0}

    # Output directories
    base_dir = config.chips_dir / str(chip_size) / pair_key
    t1_dir = base_dir / "images_t1"
    t2_dir = base_dir / "images_t2"
    mask_dir = base_dir / "masks"
    for d in [t1_dir, t2_dir, mask_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # Open all three rasters
    src_t1 = rasterio.open(t1_path)
    src_t2 = rasterio.open(t2_path)
    src_mask = rasterio.open(mask_path)

    # Use T1 as the reference grid
    h, w = src_t1.height, src_t1.width
    step = int(chip_size * (1 - config.overlap))
    n_rows = (h - chip_size) // step + 1
    n_cols = (w - chip_size) // step + 1
    total = n_rows * n_cols

    log.info(f"  Grid: {n_cols}x{n_rows} = {total} candidate chips")

    # Profiles for saving
    img_profile = {'driver': 'GTiff', 'crs': src_t1.crs}
    mask_profile = {'driver': 'GTiff', 'crs': src_mask.crs}

    stats = {
        'total_candidates': total,
        'checked': 0,
        'created': 0,
        'filtered_nodata': 0,
        'filtered_boundary': 0,
    }
    chip_index = []  # List of valid chip positions

    pbar = tqdm(total=total, desc=f"  {chip_size}px {pair_key}", leave=True)

    for ri in range(n_rows):
        for ci in range(n_cols):
            row_off = ri * step
            col_off = ci * step

            # Boundary check
            if row_off + chip_size > h or col_off + chip_size > w:
                stats['filtered_boundary'] += 1
                pbar.update(1)
                continue

            window = Window(col_off, row_off, chip_size, chip_size)
            stats['checked'] += 1

            # --- Read chips ---
            try:
                chip_t1 = src_t1.read(window=window)  # (bands, H, W)
                chip_t2 = src_t2.read(window=window)
                chip_mask = src_mask.read(1, window=window)  # (H, W)
            except Exception:
                stats['filtered_nodata'] += 1
                pbar.update(1)
                continue

            # --- Nodata filter ---
            # A pixel is valid if at least one band is non-zero in both T1 and T2
            t1_valid = np.any(chip_t1 != 0, axis=0)
            t2_valid = np.any(chip_t2 != 0, axis=0)
            combined_valid = t1_valid & t2_valid
            valid_ratio = np.sum(combined_valid) / (chip_size * chip_size)

            if valid_ratio < config.min_valid_ratio:
                stats['filtered_nodata'] += 1
                pbar.update(1)
                continue

            # --- Save chips ---
            chip_name = f"chip_r{ri:04d}_c{ci:04d}.tif"

            save_chip(chip_t1, t1_dir / chip_name, img_profile, window, src_t1.transform)
            save_chip(chip_t2, t2_dir / chip_name, img_profile, window, src_t2.transform)
            save_chip(chip_mask, mask_dir / chip_name, mask_profile, window, src_mask.transform)

            chip_index.append({
                'filename': chip_name,
                'row': ri,
                'col': ci,
                'row_off': row_off,
                'col_off': col_off,
                'valid_ratio': float(valid_ratio),
            })

            stats['created'] += 1
            pbar.update(1)

    pbar.close()
    src_t1.close()
    src_t2.close()
    src_mask.close()

    # --- Also chip secondary channels for Layout B ---
    if registry['layout'] == 'B':
        # Chip RGB, NDVI, EVI at the same positions as the valid NGB chips
        for channel in ['RGB', 'NDVI', 'EVI']:
            for year_label, year_val in [('t1', t1), ('t2', t2)]:
                if channel not in registry['imagery'].get(year_val, {}):
                    continue

                ch_path = registry['imagery'][year_val][channel]
                ch_out_dir = base_dir / f"{channel.lower()}_{year_label}"
                ch_out_dir.mkdir(parents=True, exist_ok=True)

                with rasterio.open(ch_path) as ch_src:
                    ch_profile = {'driver': 'GTiff', 'crs': ch_src.crs}
                    for entry in tqdm(
                        chip_index,
                        desc=f"    {channel} {year_label} {pair_key}",
                        leave=False,
                    ):
                        w = Window(
                            entry['col_off'], entry['row_off'],
                            chip_size, chip_size,
                        )
                        try:
                            ch_data = ch_src.read(window=w)
                            save_chip(
                                ch_data, ch_out_dir / entry['filename'],
                                ch_profile, w, ch_src.transform,
                            )
                        except Exception as e:
                            log.warning(f"    Failed {channel} {entry['filename']}: {e}")

    # --- Save chip index ---
    index_path = base_dir / "chip_index.json"
    with open(index_path, 'w') as f:
        json.dump({
            'pair': pair_key,
            'chip_size': chip_size,
            'layout': registry['layout'],
            'raster_height': h,
            'raster_width': w,
            'n_bands_t1': src_t1.count if hasattr(src_t1, 'count') else None,
            'statistics': stats,
            'chips': chip_index,
        }, f, indent=2)

    return stats

In [23]:
log.info("=" * 70)
log.info("STEP 3: Patching all year pairs")
log.info("=" * 70)

start_time = datetime.now()
all_stats = {}

for chip_size in config.chip_sizes:
    log.info(f"\nChip size: {chip_size}x{chip_size}")
    log.info("-" * 50)

    for t1, t2 in config.year_pairs:
        log.info(f"\n  Year pair: {t1} -> {t2}")
        stats = patch_year_pair(t1, t2, chip_size, registry, config)
        all_stats[(chip_size, t1, t2)] = stats

        keep_rate = stats['created'] / max(stats.get('checked', 1), 1) * 100
        log.info(
            f"  Result: {stats['created']} created, "
            f"{stats.get('filtered_nodata', 0)} filtered (nodata), "
            f"{keep_rate:.1f}% keep rate"
        )

elapsed = datetime.now() - start_time
log.info(f"\nTotal patching time: {elapsed}")

21:16:55 | INFO | ======================================================================
21:16:55 | INFO | STEP 3: Patching all year pairs
21:16:55 | INFO | ======================================================================
21:16:55 | INFO | 
Chip size: 256x256
21:16:55 | INFO | --------------------------------------------------
21:16:55 | INFO | 
  Year pair: 2021 -> 2022
21:16:55 | INFO |   Grid: 86x75 = 6450 candidate chips
  256px 2021_2022: 100%|██████████| 6450/6450 [04:15<00:00, 25.22it/s]
21:21:11 | INFO |   Result: 6450 created, 0 filtered (nodata), 100.0% keep rate
21:21:11 | INFO | 
  Year pair: 2022 -> 2023
21:21:11 | INFO |   Grid: 86x75 = 6450 candidate chips
  256px 2022_2023: 100%|██████████| 6450/6450 [04:18<00:00, 24.95it/s]
21:25:30 | INFO |   Result: 6450 created, 0 filtered (nodata), 100.0% keep rate
21:25:30 | INFO | 
  Year pair: 2023 -> 2024
21:25:30 | INFO |   Grid: 86x75 = 6450 candidate chips
  256px 2023_2024: 100%|██████████| 6450/6450 [04:18<00:00, 24.

## Summary

In [24]:
print("\n" + "=" * 70)
print("PATCHING SUMMARY")
print("=" * 70)
print(f"{'Chip Size':<12} {'Pair':<12} {'Candidates':<12} {'Created':<10} {'Keep %':<10}")
print("-" * 70)

grand_total = 0
for (cs, t1, t2), stats in all_stats.items():
    cand = stats.get('total_candidates', 0)
    created = stats['created']
    keep = created / max(stats.get('checked', 1), 1) * 100
    grand_total += created
    print(f"{cs}x{cs:<8} {t1}->{t2:<6} {cand:<12} {created:<10} {keep:<10.1f}")

print("-" * 70)
print(f"Total chips on disk: {grand_total}")
print("=" * 70)

# Show output directory structure
print(f"\nOutput directory: {config.chips_dir}")
for cs_dir in sorted(config.chips_dir.iterdir()):
    if cs_dir.is_dir():
        for pair_dir in sorted(cs_dir.iterdir()):
            if pair_dir.is_dir():
                n_files = sum(1 for _ in pair_dir.rglob('*.tif'))
                print(f"  {cs_dir.name}/{pair_dir.name}/  ({n_files} files)")


PATCHING SUMMARY
Chip Size    Pair         Candidates   Created    Keep %    
----------------------------------------------------------------------
256x256      2021->2022   6450         6450       100.0     
256x256      2022->2023   6450         6450       100.0     
256x256      2023->2024   6450         6450       100.0     
512x512      2021->2022   1591         1591       100.0     
512x512      2022->2023   1591         1591       100.0     
512x512      2023->2024   1591         1591       100.0     
----------------------------------------------------------------------
Total chips on disk: 24123

Output directory: Dataset\GEE_Kalimantan_Chips
  256/2021_2022/  (19350 files)
  256/2022_2023/  (19350 files)
  256/2023_2024/  (19350 files)
  512/2021_2022/  (4773 files)
  512/2022_2023/  (4773 files)
  512/2023_2024/  (4773 files)
